In [1]:
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    content: str
    feedback: str
    revision_count: int
    is_approved: bool

In [2]:
# 节点 1: 生成器
def generator(state: AgentState):
    print("---执行生成节点---")
    return {"content": "这是初始生成的文案内容。", "revision_count": state.get("revision_count", 0) + 1}

# 节点 2: 审核器 (逻辑控制中心)
def evaluator(state: AgentState):
    print("---执行审核节点---")
    # 模拟精细控制逻辑：如果修改超过2次则强制通过，否则检查是否有反馈
    if state["revision_count"] >= 3:
        return {"is_approved": True}
    return {"is_approved": False, "feedback": "内容不够生动，请重新优化。"}

# 路由逻辑：决定流转方向
def decide_to_go(state: AgentState):
    if state["is_approved"]:
        return "end"
    return "generate"

In [3]:
workflow = StateGraph(AgentState)

# 添加节点
workflow.add_node("generate", generator)
workflow.add_node("evaluate", evaluator)

# 设置入口
workflow.set_entry_point("generate")

# 节点间连线
workflow.add_edge("generate", "evaluate")

# 动态路由控制 (精细化的分支逻辑)
workflow.add_conditional_edges(
    "evaluate",
    decide_to_go,
    {
        "generate": "generate",
        "end": END
    }
)

# 编译
app = workflow.compile()

In [4]:
import operator
from typing import Annotated, TypedDict, Union
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# --- 1. 定义状态 ---
class AgentState(TypedDict):
    content: str
    approved: bool

# --- 2. 定义节点逻辑 ---
def designer_node(state: AgentState):
    print("🎨 [Designer]: 正在生成文案...")
    return {"content": "这是一条由 AI 生成的初始广告文案。", "approved": False}

def manager_node(state: AgentState):
    # 这个节点可以作为逻辑判断，也可以只是一个占位符，等待人工修改它的 approved 状态
    print("📋 [Manager]: 正在检查文案合规性...")
    return state

# --- 3. 构建图 ---
workflow = StateGraph(AgentState)

workflow.add_node("designer", designer_node)
workflow.add_node("manager", manager_node)

workflow.set_entry_point("designer")
workflow.add_edge("designer", "manager")

# 精细控制：根据 approved 字段决定结束还是重写
workflow.add_conditional_edges(
    "manager",
    lambda x: "end" if x["approved"] else "designer",
    {"end": END, "designer": "designer"}
)

# --- 4. 关键：添加内存检查点并设置断点 ---
memory = MemorySaver()
# 我们在 manager 节点执行前设置中断，进行“人工干预”
app = workflow.compile(checkpointer=memory, interrupt_before=["manager"])

# --- 5. 运行演示 ---
config = {"configurable": {"thread_id": "1"}}

print("\n--- 第一次启动：运行到断点 ---")
for event in app.stream({"content": "开始", "approved": False}, config):
    print(event)

# 此时程序会暂停在 manager 节点之前
snapshot = app.get_state(config)
print(f"\n当前状态 (暂停中): {snapshot.values}")

print("\n--- 人工干预：手动修改状态为 '已批准' ---")
# 模拟人工通过：将 approved 改为 True
app.update_state(config, {"approved": True}, as_node="manager")

print("\n--- 继续执行：从断点恢复 ---")
for event in app.stream(None, config): # 传入 None 表示从检查点恢复
    print(event)

print("\n--- 流程结束 ---")


--- 第一次启动：运行到断点 ---
🎨 [Designer]: 正在生成文案...
{'designer': {'content': '这是一条由 AI 生成的初始广告文案。', 'approved': False}}
{'__interrupt__': ()}

当前状态 (暂停中): {'content': '这是一条由 AI 生成的初始广告文案。', 'approved': False}

--- 人工干预：手动修改状态为 '已批准' ---

--- 继续执行：从断点恢复 ---

--- 流程结束 ---
